# TTS Experiment — Facebook MMS TTS

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Evaluate Facebook's Massively Multilingual Speech (MMS) TTS models as a replacement
for Coqui XTTS v2. MMS provides **native** Cebuano and Tagalog models — no Spanish
phoneme approximation required.

### Models used
| Language | Model | Params |
|---|---|---|
| Cebuano | `facebook/mms-tts-ceb` | 36.3M |
| Tagalog | `facebook/mms-tts-tgl` | 36.3M |
| English | `facebook/mms-tts-eng` | 36.3M |

### Bulletin
`PAGASA_20-19W_Pepito_SWB#01` — Severe Weather Bulletin, Tropical Depression Pepito

### Synthesis order
CEB → TL → EN

### License note
MMS TTS models are CC-BY-NC 4.0 (non-commercial). Acceptable for this hackathon.
Flag for production licensing review.

In [ ]:
import re
import time
import numpy as np
import torch
from pathlib import Path
from pydub import AudioSegment
from transformers import VitsModel, AutoTokenizer

# --- Paths ---
notebook_dir = Path(".")
output_dir = notebook_dir / "08-mms-tts-experiment"
output_dir.mkdir(exist_ok=True)

data_dir = notebook_dir.parent / "data"
input_dir = data_dir / "radio_bulletins"

# --- Experiment scope ---
STEM = "PAGASA_20-19W_Pepito_SWB#01"
LANGUAGES = ["ceb", "tl", "en"]  # synthesis order: CEB first
MMS_MODELS = {
    "ceb": "facebook/mms-tts-ceb",
    "tl":  "facebook/mms-tts-tgl",
    "en":  "facebook/mms-tts-eng",
}

# --- Verify input files exist ---
input_files = {lang: input_dir / f"{STEM}_radio_{lang}.md" for lang in LANGUAGES}
missing = [str(p) for p in input_files.values() if not p.exists()]
if missing:
    print(f"⚠  Missing input files: {missing}")
else:
    print(f"✓ All 3 input files found")
    for lang, p in input_files.items():
        print(f"  {lang}: {p.name}")

print(f"✓ Output dir: {output_dir.absolute()}")

## 1. Text Preprocessing

Reuse `preprocess_for_tts` from notebook 07 — strips markdown formatting so
nothing is read aloud except actual spoken content.

In [ ]:
def preprocess_for_tts(markdown_text: str) -> str:
    """Strip markdown formatting for clean TTS input.

    Nothing in the output should be read aloud unless it is actual spoken content.
    Modal-ready: pure function, no external state.
    Copied from notebook 07 — keep in sync if updated there.
    """
    text = markdown_text

    # Remove stage directions: **(Sound effect: ...)** on their own line
    text = re.sub(r"^\s*\*\*\([^)]+\)\*\*\s*$", "", text, flags=re.MULTILINE)

    # Remove role labels: **BROADCASTER:** **Boses:** **LABEL:** etc.
    text = re.sub(r"\*\*[A-Za-z][A-Za-z\s]+:\*\*\s*", "", text)

    # Section headings → just the heading text as a plain spoken sentence
    text = re.sub(r"^#{1,6}\s+(.+)$", r"\1.", text, flags=re.MULTILINE)

    # Horizontal rules — must come BEFORE bold/italic removal to avoid *** corruption
    text = re.sub(r"^[-*_]{3,}$", "", text, flags=re.MULTILINE)

    # Bold and italic markers
    text = re.sub(r"\*{1,3}(.+?)\*{1,3}", r"\1", text)
    text = re.sub(r"_{1,3}(.+?)_{1,3}", r"\1", text)

    # Inline code
    text = re.sub(r"`(.+?)`", r"\1", text)

    # Blockquotes
    text = re.sub(r"^>\s+", "", text, flags=re.MULTILINE)

    # Numbered and bulleted list markers
    text = re.sub(r"^\s*\d+\.\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[-*+]\s+", "", text, flags=re.MULTILINE)

    # Remove any leftover parenthetical stage directions (fallback, after bold stripped)
    text = re.sub(r"^\s*\([^)]{10,}\)\s*$", "", text, flags=re.MULTILINE)

    # Remove role labels that survived bold stripping: WORD: at start of line
    text = re.sub(r"^[A-Z][A-Za-z\s]+:\s*$", "", text, flags=re.MULTILINE)

    # Collapse 3+ blank lines to 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Preprocess all 3 scripts upfront and save plain text files
plain_texts = {}
for lang in LANGUAGES:
    md = input_files[lang].read_text(encoding="utf-8")
    plain = preprocess_for_tts(md)
    plain_texts[lang] = plain

    plain_path = output_dir / f"{STEM}_radio_{lang}_plain.txt"
    plain_path.write_text(plain, encoding="utf-8")
    print(f"✓ {lang}: {len(md)} chars → {len(plain)} chars  (saved {plain_path.name})")

# Preview CEB plain text
print("\nPREPROCESSED TEXT — CEB (first 500 chars)")
print("=" * 60)
print(plain_texts["ceb"][:500])
print("...")

## 2. Load MMS Models

Load all three VITS models simultaneously. Each is ~140 MB (F32). Downloads are cached
by HuggingFace after first run.

Unlike XTTS v2 (1.8 GB single multilingual model), MMS uses one small model per language.

In [ ]:
print("Loading MMS models (downloads ~140 MB each on first run, cached after)...")
t0 = time.time()

models = {}
tokenizers = {}

for lang in LANGUAGES:
    model_id = MMS_MODELS[lang]
    lang_label = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}[lang]
    print(f"  Loading {lang_label} ({model_id})...")
    t_lang = time.time()
    tokenizers[lang] = AutoTokenizer.from_pretrained(model_id)
    models[lang] = VitsModel.from_pretrained(model_id)
    models[lang].eval()
    print(f"  ✓ {lang_label} loaded in {time.time() - t_lang:.1f}s")

print(f"\n✓ All 3 models loaded in {time.time() - t0:.1f}s total")
print(f"  Sample rates: { {lang: models[lang].config.sampling_rate for lang in LANGUAGES} }")